In [ ]:
import json
import os
import tempfile
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from json import JSONDecodeError
# import importlib
from models import models
from data_cleaner import create_dataframe


In [ ]:
system_prompt = """
You are an agent that generates realistic synthetic datasets.
you must respond with only a valid JSON list of objects, no commentary.
response format must = json_object. Don't forget this.
"""

In [ ]:
user_prompt = """
Create 5 rows of synthetic data for customer support tickets at a Blockchain company.
Each row should have these fields: ticket_id, customer_name, issue, urgency, status.
Vary the names, issues, and urgency levels so the rows are diverse.
"""

In [ ]:
# Print the available models for the user and let them select, then save it in 'choice' variable

# print(f"Available Models:")

# for name in models:
#     print(name)

# choice = input("Choose a model to proceed")




# use the saved model name in choice to call the selected function

# if choice in models:
#     answer = models[choice](system_prompt, user_prompt)
#     print(answer)
# else:
#     print("Model not available.")

# try:
#     df = create_dataframe(answer)
#     print(df)
# except json.JSONDecodeError as e:
#     print(e, repr(answer[:50]))




# Remember, all the code here would work manually, but i'm commenting it because i'm building a new function to be connected to gradio


In [ ]:
# Build the function for gradio

def generate_data(user_prompt, model_name):
    # This one selects the model via a dropdown in gradio
    answer = models[model_name](system_prompt, user_prompt)

    try:
        # This cleans the response and converts it to a DataFrame
        df = create_dataframe(answer)

        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".csv",
            delete=False,
            encoding="utf-8",
            newline=""
        ) as file:

            df.to_csv(file.name, index=False)
            csv_path = file.name

        # return both the DataFrame and csv file path
        return df, csv_path

    except json.JSONDecodeError as e:
        raise gr.Error(f"Could not process model output {e}")

In [ ]:
# And now, the gradio function finally

from openai.types.responses.computer_action import Click


with gr.Blocks() as demo:
    gr.Markdown("# Synthetic Data Generator")

    user_input = gr.Textbox(label="What Synthetic Data Do You Want to Generate?", lines=3)

    model_dropdown = gr.Dropdown(
        choices= list(models.keys()),
        value="gpt",
        label="Select a Model")

    generate_button = gr.Button("Generate Data")

    output = gr.DataFrame(label="Generated Data")

    csv_download = gr.File(label="Download CSV")

    generate_button.click(
        fn=generate_data,
        inputs=[user_input, model_dropdown],
        outputs=[output, csv_download]
    )

    demo.launch(inbrowser=True)